In [1]:
# src/utils

import sys
sys.path.append("/home/jovyan/work")

from src.utils.spark_session import get_spark_session
spark = get_spark_session("generate-sample-data")
spark.sql("CREATE DATABASE IF NOT EXISTS silver")

DataFrame[]

In [3]:
# Reference/ congig tables

from pyspark.sql import Row
from pyspark.sql.types import StructType, StructField, LongType, StringType, DateType
from datetime import datetime

now = datetime.now()

# --- branch ---
df_branch = spark.createDataFrame([
    Row(branch_id=1, name="Lagos - Head Office", address="Ikeja, Lagos",
        is_active=True, created_at=now, updated_at=now),
])
df_branch.write.format("delta").mode("overwrite").saveAsTable("silver.branch")

# --- data_source ---
df_data_source = spark.createDataFrame([
    Row(source_id=1, source_code="GOOGLE_FORM", description="Google Forms submission"),
    Row(source_id=2, source_code="WHATSAPP_GROUP_AGENT", description="WhatsApp group message, LLM-extracted"),
    Row(source_id=3, source_code="RECEIPT_AGENT", description="WhatsApp receipt photo, OCR-extracted"),
    Row(source_id=4, source_code="MANUAL", description="Manual entry / correction"),
])
df_data_source.write.format("delta").mode("overwrite").saveAsTable("silver.data_source")

# --- auction_house ---
df_auction_house = spark.createDataFrame([
    Row(auction_house_id=1, name="Copart", country="USA", created_at=now),
    Row(auction_house_id=2, name="IAAI", country="USA", created_at=now),
    Row(auction_house_id=3, name="Manheim", country="USA", created_at=now),
])
df_auction_house.write.format("delta").mode("overwrite").saveAsTable("silver.auction_house")

# --- vendor ---
df_vendor = spark.createDataFrame([
    Row(vendor_id=1, name="AutoParts Lagos Ltd", contact_phone="+2348012345001", contact_email="sales@autopartslagos.ng", created_at=now),
    Row(vendor_id=2, name="Naija Spares Hub", contact_phone="+2348012345002", contact_email="info@naijaspares.ng", created_at=now),
    Row(vendor_id=3, name="Trust Motor Spares", contact_phone="+2348012345003", contact_email=None, created_at=now),
])
df_vendor.write.format("delta").mode("overwrite").saveAsTable("silver.vendor")

# --- part ---
df_part = spark.createDataFrame([
    Row(part_id=1, name="Front Bumper", category="Body", created_at=now),
    Row(part_id=2, name="Headlight Assembly", category="Body", created_at=now),
    Row(part_id=3, name="Side Mirror", category="Body", created_at=now),
    Row(part_id=4, name="Windshield", category="Body", created_at=now),
    Row(part_id=5, name="Brake Pads", category="Mechanical", created_at=now),
    Row(part_id=6, name="Alternator", category="Electrical", created_at=now),
    Row(part_id=7, name="Radiator", category="Mechanical", created_at=now),
    Row(part_id=8, name="Seat Cover", category="Interior", created_at=now),
])
df_part.write.format("delta").mode("overwrite").saveAsTable("silver.part")

# --- staff ---
df_staff = spark.createDataFrame([
    Row(staff_id=1, branch_id=1, name="Tunde Bakare", phone_whatsapp="+2348030000001", is_active=True, created_at=now, updated_at=now),
    Row(staff_id=2, branch_id=1, name="Chinedu Okafor", phone_whatsapp="+2348030000002", is_active=True, created_at=now, updated_at=now),
    Row(staff_id=3, branch_id=1, name="Musa Ibrahim", phone_whatsapp="+2348030000003", is_active=True, created_at=now, updated_at=now),
    Row(staff_id=4, branch_id=1, name="Ifeoma Nwosu", phone_whatsapp="+2348030000004", is_active=True, created_at=now, updated_at=now),
    Row(staff_id=5, branch_id=1, name="Segun Adeyemi", phone_whatsapp="+2348030000005", is_active=True, created_at=now, updated_at=now),
    Row(staff_id=6, branch_id=1, name="Amaka Eze", phone_whatsapp="+2348030000006", is_active=True, created_at=now, updated_at=now),
])
df_staff.write.format("delta").mode("overwrite").saveAsTable("silver.staff")

# --- staff_role (who does what — matches "roles not fixed headcount" from Phase 1) ---
staff_role_schema = StructType([
    StructField("staff_id", LongType(), False),
    StructField("role_code", StringType(), False),
    StructField("valid_from", DateType(), False),
    StructField("valid_to", DateType(), True),
])

df_staff_role = spark.createDataFrame([
    Row(staff_id=1, role_code="OWNER", valid_from=now.date(), valid_to=None),
    Row(staff_id=1, role_code="FINANCE", valid_from=now.date(), valid_to=None),
    Row(staff_id=2, role_code="DRIVER", valid_from=now.date(), valid_to=None),
    Row(staff_id=3, role_code="DRIVER", valid_from=now.date(), valid_to=None),
    Row(staff_id=4, role_code="INSPECTOR", valid_from=now.date(), valid_to=None),
    Row(staff_id=4, role_code="MECHANIC", valid_from=now.date(), valid_to=None),
    Row(staff_id=5, role_code="MECHANIC", valid_from=now.date(), valid_to=None),
    Row(staff_id=6, role_code="SALES", valid_from=now.date(), valid_to=None),
], schema=staff_role_schema)
df_staff_role.write.format("delta").mode("overwrite").saveAsTable("silver.staff_role")

# df_staff_role = spark.createDataFrame([
#     Row(staff_id=1, role_code="OWNER", valid_from=now.date(), valid_to=None),
#     Row(staff_id=1, role_code="FINANCE", valid_from=now.date(), valid_to=None),
#     Row(staff_id=2, role_code="DRIVER", valid_from=now.date(), valid_to=None),
#     Row(staff_id=3, role_code="DRIVER", valid_from=now.date(), valid_to=None),
#     Row(staff_id=4, role_code="INSPECTOR", valid_from=now.date(), valid_to=None),
#     Row(staff_id=4, role_code="MECHANIC", valid_from=now.date(), valid_to=None),
#     Row(staff_id=5, role_code="MECHANIC", valid_from=now.date(), valid_to=None),
#     Row(staff_id=6, role_code="SALES", valid_from=now.date(), valid_to=None),
# ])
# df_staff_role.write.format("delta").mode("overwrite").saveAsTable("silver.staff_role")

# --- inspection_checklist_item ---
df_checklist = spark.createDataFrame([
    Row(item_id=1, category="ENGINE", label="Engine starts and runs smoothly", is_active=True, sort_order=1),
    Row(item_id=2, category="ENGINE", label="No visible oil/coolant leaks", is_active=True, sort_order=2),
    Row(item_id=3, category="BODY", label="No major dents or rust", is_active=True, sort_order=3),
    Row(item_id=4, category="BODY", label="Panel alignment is correct", is_active=True, sort_order=4),
    Row(item_id=5, category="ELECTRICAL", label="All lights functional", is_active=True, sort_order=5),
    Row(item_id=6, category="ELECTRICAL", label="AC/heating works", is_active=True, sort_order=6),
    Row(item_id=7, category="INTERIOR", label="Seats and upholstery in good condition", is_active=True, sort_order=7),
    Row(item_id=8, category="TIRES", label="Tire tread depth acceptable", is_active=True, sort_order=8),
])
df_checklist.write.format("delta").mode("overwrite").saveAsTable("silver.inspection_checklist_item")

for t in ["branch", "data_source", "auction_house", "vendor", "part", "staff", "staff_role", "inspection_checklist_item"]:
    print(f"silver.{t}: {spark.table(f'silver.{t}').count()} rows")

silver.branch: 1 rows
silver.data_source: 4 rows
silver.auction_house: 3 rows
silver.vendor: 3 rows
silver.part: 8 rows
silver.staff: 6 rows
silver.staff_role: 8 rows
silver.inspection_checklist_item: 8 rows
